# Test a custom prompt template

**What this does:** takes a candidate LLM prompt template, runs it against a sample of historical decisions from the operator's `LlmDecision` table, and shows how its outputs compare against the actual recorded decisions.

**What it doesn't do:** open trades, modify any row, or replace a live prompt. Editing `crypto/strategy.py` to use the new prompt is a separate, explicit step that the operator does after this notebook gives them confidence.

Halal alignment: research surface only. The candidate prompt's outputs are scored against the historical record but never executed.

## 0 · Imports + connection

Same pattern as `explore-replay-store.ipynb`. Reads `DATABASE_URL` from `.env`.

In [ ]:
from __future__ import annotations

from datetime import datetime, timedelta, UTC

import asyncpg
import pandas as pd

from halal_trader.config import get_settings
from halal_trader.core.llm.factory import create_llm

settings = get_settings()
DATABASE_URL = settings.database_url

## 1 · Pick a sample of historical decisions

We use the recorded `prompt_summary` field as the input we want to re-run. Pin: the field strips operator-specific identifiers (cycle_id, balances) so feeding it back into the LLM is safe.

**Tunable:** `SAMPLE_SIZE` controls how many decisions to re-test. Start small (10) to keep cost down.

In [ ]:
SAMPLE_SIZE = 10
LOOKBACK_DAYS = 30

async def fetch_sample() -> pd.DataFrame:
    conn = await asyncpg.connect(DATABASE_URL)
    try:
        rows = await conn.fetch(
            """
            SELECT id, timestamp, prompt_summary, parsed_action,
                   provider, model, prompt_version
            FROM llm_decisions
            WHERE timestamp >= $1
              AND prompt_summary IS NOT NULL
            ORDER BY random()
            LIMIT $2
            """,
            datetime.now(UTC) - timedelta(days=LOOKBACK_DAYS),
            SAMPLE_SIZE,
        )
        return pd.DataFrame([dict(r) for r in rows])
    finally:
        await conn.close()

sample = await fetch_sample()
print(f"Sampled {len(sample)} decisions")
sample[["timestamp", "provider", "model", "prompt_version"]].head()

## 2 · Define the candidate prompt

Edit the cell below. The candidate prompt should accept the same `prompt_summary` shape the bot already produces — operators iterating on a new system prompt only need to change the system_prompt and validation; the user prompt comes from the historical row.

Pin: registering the candidate prompt with `core.llm.prompts.register` is **not** done here. This notebook is for ad-hoc testing; once the operator commits to a candidate, they bump the version + register it via the normal path.

In [ ]:
CANDIDATE_SYSTEM_PROMPT = """
You are a halal crypto-trading analyst. The operator gives you the
current market state and a list of pairs. For each pair, decide BUY
/ SELL / HOLD with a confidence ∈ [0, 1] and quantity in base
currency.

EDIT-ME: tighten the entry filter — refuse BUY when RSI > 70.

Output strict JSON: {"decisions": [...], "market_outlook": "..."}
""".strip()

## 3 · Run the candidate against each sample

We use the bot's own `core.llm.factory:create_llm` to ensure the candidate runs through the same provider chain (with cost capping + circuit breaker) the live cycle uses. Pin: the LLM cost cap (`LLM_DAILY_USD_CAP`) still applies to this notebook's calls; if you've nearly maxed the cap on the day's live trading, this run will hit it.

Each row's `candidate_response` is the raw string the candidate prompt produced. Comparison is in the next cell.

In [ ]:
llm = create_llm(settings)

candidate_responses: list[str] = []
for _, row in sample.iterrows():
    response = await llm.generate(
        prompt=row["prompt_summary"],
        system=CANDIDATE_SYSTEM_PROMPT,
    )
    candidate_responses.append(response)

sample["candidate_response"] = candidate_responses
print("Candidate run complete; cost charged to the daily LLM cap.")

## 4 · Compare candidate vs recorded

Naive but effective: count how many decisions agree on the per-pair `action` field. If the candidate prompt produces materially different decisions, this number will drop. A useful starting bar is 80% agreement — below that, investigate why before promoting.

In [ ]:
import json

def parse_actions(blob) -> dict[str, str]:
    """Return {pair: action} from either a dict (parsed_action) or a JSON string (candidate response)."""
    if isinstance(blob, dict):
        decisions = blob.get("decisions") or []
    else:
        try:
            decisions = json.loads(blob).get("decisions") or []
        except Exception:
            return {}
    out = {}
    for d in decisions:
        out[str(d.get("pair") or d.get("symbol"))] = str(d.get("action")).lower()
    return out

agree = 0
total = 0
for _, row in sample.iterrows():
    recorded = parse_actions(row["parsed_action"])
    candidate = parse_actions(row["candidate_response"])
    for pair, action in recorded.items():
        total += 1
        if candidate.get(pair) == action:
            agree += 1

if total == 0:
    print("No comparable decisions in the sample.")
else:
    print(f"Agreement: {agree}/{total} = {agree/total:.0%}")

## 5 · Promote-to-live checklist

If the agreement rate looks reasonable AND the candidate's divergences are intentional (e.g., the new RSI filter rejects exactly the buys the operator wanted to filter), the next steps are:

1. Move the candidate prompt into a Python module under `src/halal_trader/core/llm/prompts/`.
2. Register it via `register(name="crypto.strategy.v2", template=...)` so future `LlmDecision` rows record the new prompt_version.
3. Run the **Wave 5.B A/B comparator** (`core/ab_compare.py`) against the live data once 100+ trades have run on the new prompt.
4. Run the **Wave 4.F promotion gate** (`core/promotion_gate.py`) to verify the live numbers clear the absolute thresholds.
5. Run the **Wave 6.D ML CI pipeline** (`ml/ci_pipeline.py`) to verify no regressions vs the incumbent.

Only after steps 3-5 pass should the operator make the new prompt the production default in `crypto/strategy.py`.